# Missing Data Handling — A Hands-On ML Course

**Beginner-friendly project covering:** MCAR / MAR / MNAR missingness mechanisms,
SimpleImputer, KNNImputer, IterativeImputer (MICE), missingness indicators,
leakage-safe train/test workflows, model comparison, and production pipelines.

Run every cell top to bottom. Each section starts with a markdown explanation,
followed by code, with comments explaining what each part does.


## Section 0 — Generate the dataset

We're simulating a small health dataset with three *intentionally different*
types of missingness, so we can compare how each behaves:

- `bmi` → missing **completely at random** (MCAR)
- `glucose` → missing depending on `income_bracket` (MAR)
- `blood_pressure` → missing depending on its **own** high values (MNAR)

**Note on `has_condition`:** it's generated with `np.random.binomial`,
completely independent of every other column. That's deliberate — this
notebook is about *imputation mechanics*, not about building a strong
classifier. Don't be surprised if AUC in Section 10/11 lands close to (or
even below) 0.5 — there's no real signal to find. If you want to see
imputation choices actually move a metric, try the stretch challenge of
making `has_condition` depend on `bmi`/`glucose`/`blood_pressure`.


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(7)
n = 800

df = pd.DataFrame({
    "age": np.random.randint(20, 80, n),
    "bmi": np.random.normal(26, 5, n),
    "blood_pressure": np.random.normal(120, 18, n),
    "glucose": np.random.normal(95, 20, n),
    "smoking_status": np.random.choice(
        ["never", "former", "current"], n
    ),
    "income_bracket": np.random.choice(
        ["low", "mid", "high"], n
    ),
    "hospital_visits": np.random.poisson(1.5, n),
    "has_condition": np.random.binomial(1, 0.25, n)
})

# MCAR: 12% of bmi values vanish completely at random
mask_mcar = np.random.rand(n) < 0.12
df.loc[mask_mcar, "bmi"] = np.nan

# MAR: glucose is more likely to be missing for low-income patients
mask_mar = df["income_bracket"] == "low"
df.loc[
    mask_mar & (np.random.rand(n) < 0.35),
    "glucose"
] = np.nan

# MNAR: high blood pressure readings are more likely to go missing
mask_mnar = df["blood_pressure"] > 145
df.loc[
    mask_mnar & (np.random.rand(n) < 0.4),
    "blood_pressure"
] = np.nan

df.to_csv("day02_health.csv", index=False)
df.head()


## Section 1 — Initial Data Exploration

**Concept:** Before touching missing values, get a full picture of the
dataset's shape, types, distributions, and how much is actually missing.

**Why it matters:** You can't choose a sensible imputation strategy without
knowing how much is missing, in which columns, and what the rest of the data
looks like.


In [ ]:
# Dimensions: rows x columns
print("Shape:", df.shape)


In [ ]:
# Data types and non-null counts
df.info()


In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)


In [ ]:
# Raw missing counts
df.isnull().sum()


In [ ]:
# Missing value percentages — sorted, only columns with gaps
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
missing_pct[missing_pct > 0].sort_values(ascending=False)


In [ ]:
# Class balance for the target variable
df["has_condition"].value_counts(normalize=True).round(2)


In [ ]:
# Categorical feature distributions
print(df["smoking_status"].value_counts(normalize=True).round(2))
print()
print(df["income_bracket"].value_counts(normalize=True).round(2))


**Interpretation:** You should see `bmi` (~12%), `glucose` (~8%), and
`blood_pressure` (~3%) missing, with `has_condition` imbalanced at roughly
75/25. Keep that imbalance in mind — accuracy alone will be misleading later.

**Common mistake:** treating all missing columns the same regardless of how
much is missing or why.

**Reflection:**
- Which column has the most missing data?
- Why might low income correlate with missing glucose readings?
- How should the 25% positive rate change how you evaluate models later?


## Section 2 — Understanding Missingness Mechanisms

**Concept:** Missing data usually has structure. There are three canonical
mechanisms:

| Type | Definition | Example | Difficulty |
|------|------------|---------|------------|
| **MCAR** | Missingness is unrelated to any variable, observed or missing | A lab accidentally loses 10% of samples at random | Easiest |
| **MAR** | Missingness depends on *observed* variables | Low-income patients skip glucose tests due to access | Medium |
| **MNAR** | Missingness depends on the *missing value itself* | People with high blood pressure avoid getting it measured | Hardest |

**Why it matters:** The mechanism determines whether an imputation method
will introduce bias. MCAR is "safe" for any method. MAR requires modeling
relationships between features. MNAR can't be fully fixed by imputation —
it requires domain knowledge.

**Why no test can prove MAR or MNAR:** To prove a value's missingness
depends on itself, you'd need to know the value — but it's missing. You can
build circumstantial evidence (visualizations, group comparisons, domain
knowledge), but never proof.


## Section 3 — Visualizing Missingness

**Concept:** Visualizations reveal missingness patterns that summary
statistics hide — clustering, correlation between missing columns, and
alignment with other variables.

If `missingno` isn't installed, run `pip install missingno` first.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False
    print("missingno not installed — run: pip install missingno")


In [ ]:
if HAS_MSNO:
    # White = missing, dark = present. Look for aligned vertical stripes.
    msno.matrix(df, figsize=(10, 5))
    plt.title("Missingness matrix")
    plt.tight_layout()
    plt.savefig("missingness_matrix.png", dpi=120)
    plt.show()


In [ ]:
if HAS_MSNO:
    # How much is missing per column
    msno.bar(df, figsize=(10, 4))
    plt.title("Completeness per column")
    plt.tight_layout()
    plt.savefig("missingness_bar.png", dpi=120)
    plt.show()


In [ ]:
if HAS_MSNO:
    # Do columns tend to be missing together?
    msno.heatmap(df, figsize=(8, 5))
    plt.title("Missingness correlation heatmap")
    plt.tight_layout()
    plt.savefig("missingness_heatmap.png", dpi=120)
    plt.show()


In [ ]:
# Custom seaborn heatmap of the missingness mask itself
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False, ax=ax)
ax.set_title("Missing value heatmap (bright = missing)")
plt.tight_layout()
plt.savefig("missing_heatmap_sns.png", dpi=120)
plt.show()


**Interpretation:**
- `bmi` gaps should look scattered with no pattern (MCAR)
- `glucose` gaps may cluster more if you sort by `income_bracket`
- `blood_pressure` gaps are subtle to see directly since the high values
  that triggered missingness are now gone

**Common mistake:** stopping at the bar chart. It shows *how much* is
missing, not *why* — always check the matrix and correlation heatmap too.


## Section 4 — Investigating Missingness Types

**Concept:** Turn visual hunches into evidence using indicator columns,
crosstabs, and correlations.


In [ ]:
# Step 1: create missingness indicator columns (1 = missing, 0 = present)
df["bmi_missing"] = df["bmi"].isna().astype(int)
df["glucose_missing"] = df["glucose"].isna().astype(int)
df["bp_missing"] = df["blood_pressure"].isna().astype(int)


In [ ]:
# Test 1: is glucose missingness associated with income_bracket? (MAR check)
glucose_by_income = pd.crosstab(
    df["income_bracket"],
    df["glucose_missing"],
    normalize="index"
).round(2)
glucose_by_income


In [ ]:
# Test 2: is bmi missingness associated with income (MCAR check — should be flat)
bmi_by_income = pd.crosstab(
    df["income_bracket"],
    df["bmi_missing"],
    normalize="index"
).round(2)
print(bmi_by_income)

# Correlation between bmi missingness and other numeric features
for col in ["age", "hospital_visits"]:
    corr = df[col].corr(df["bmi_missing"])
    print(f"{col} corr with bmi_missing: {corr:.3f}")


In [ ]:
# Test 3: indirect clues for MNAR in blood_pressure, via proxy variables
bp_present_age = df[df["bp_missing"] == 0]["age"]
bp_missing_age  = df[df["bp_missing"] == 1]["age"]
print("Mean age when BP present:", round(bp_present_age.mean(), 1))
print("Mean age when BP missing:", round(bp_missing_age.mean(), 1))

print()
print(df.groupby("bp_missing")["hospital_visits"].mean().round(2))


**Interpretation:** You should see glucose missing far more often
(~30-35%) for `income_bracket == "low"` than for `mid` or `high` — that's
MAR evidence. `bmi` missingness should look roughly flat across income
groups and near-zero correlated with age/visits — consistent with MCAR.

**MNAR limitation:** we can never directly compare missing blood pressure
values to observed ones, since they're gone — we can only reason from
proxies and domain knowledge.

**Common mistake:** dropping the indicator columns after this analysis.
Keep them — they become real model features in Section 9.


## Section 5 — Train-Test Split and Leakage Prevention

**Concept:** Data leakage happens when information from the test set
influences how training data is processed — for example, fitting an
imputer's mean on the *full* dataset before splitting.

**Wrong workflow:** fit imputer on full data → split → leakage.
**Correct workflow:** split → fit imputer on `X_train` only → transform
both `X_train` and `X_test` with that fitted imputer.


In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ["age", "bmi", "blood_pressure", "glucose",
                "smoking_status", "income_bracket", "hospital_visits"]

X = df[feature_cols].copy()
y = df["has_condition"]

# stratify=y keeps the ~25% positive rate consistent in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.2f}")
print(f"Test  positive rate: {y_test.mean():.2f}")


**Reflection:**
- Why would skipping `stratify=y` risk an unbalanced split?
- Does the leakage problem get worse or better with a smaller dataset?
- Why does sklearn's `Pipeline` (Section 11) solve this automatically?


## Section 6 — Simple Imputation

**Concept:**
- **Mean** imputation assumes the feature is roughly symmetric/normal.
- **Median** imputation is robust to skew and outliers.
- **Most frequent** is the standard default for categorical columns.

**Disadvantages:** all of these reduce variance and can distort
relationships between features — every imputed row pulls toward one
fixed value.


In [ ]:
num_cols = ["age", "bmi", "blood_pressure", "glucose", "hospital_visits"]
cat_cols = ["smoking_status", "income_bracket"]


In [ ]:
from sklearn.impute import SimpleImputer

# Mean imputer — fit ONLY on training data
mean_imp = SimpleImputer(strategy="mean")
mean_imp.fit(X_train[num_cols])

X_train_mean = X_train.copy()
X_test_mean  = X_test.copy()

X_train_mean[num_cols] = mean_imp.transform(X_train[num_cols])
X_test_mean[num_cols]  = mean_imp.transform(X_test[num_cols])

print("Remaining NaNs in train numeric cols:", X_train_mean[num_cols].isnull().sum().sum())
print("Imputed mean for bmi:", round(mean_imp.statistics_[num_cols.index("bmi")], 2))


In [ ]:
# Most-frequent imputer for categoricals
cat_imp = SimpleImputer(strategy="most_frequent")
cat_imp.fit(X_train[cat_cols])

X_train_mean[cat_cols] = cat_imp.transform(X_train[cat_cols])
X_test_mean[cat_cols]  = cat_imp.transform(X_test[cat_cols])

X_train_mean.isnull().sum()


**Common mistake:** using mean imputation on `hospital_visits`
(Poisson-distributed, right-skewed) — median is more appropriate there.

**Reflection:**
- Since `blood_pressure`'s missing values are disproportionately the
  *high* ones (MNAR), does mean imputation over- or under-estimate the
  true mean?
- What happens to a BMI histogram after mean imputation?


## Section 7 — KNN Imputation

**Concept:** Instead of one global statistic, look at the *k* most similar
rows (by other features) and use their values. This captures relationships
between features that a single mean/median cannot.

**Two required preprocessing steps:**
1. **Encode categoricals** — distance calculations need numbers.
2. **Scale features** — without scaling, large-range features (like age)
   dominate the distance calculation.


In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Step 1: encode categoricals (fit on train only)
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
enc.fit(X_train[cat_cols])

X_train_knn = X_train.copy()
X_test_knn  = X_test.copy()
X_train_knn[cat_cols] = enc.transform(X_train[cat_cols])
X_test_knn[cat_cols]  = enc.transform(X_test[cat_cols])

# Step 2: scale all features (fit on train only)
scaler = StandardScaler()
scaler.fit(X_train_knn)

X_train_scaled = scaler.transform(X_train_knn)
X_test_scaled  = scaler.transform(X_test_knn)

# Step 3: KNN imputation
knn_imp = KNNImputer(n_neighbors=5, weights="distance")
knn_imp.fit(X_train_scaled)

X_train_knn_imp = knn_imp.transform(X_train_scaled)
X_test_knn_imp  = knn_imp.transform(X_test_scaled)

# Step 4: reverse the scaling to get back to original units
X_train_knn_final = pd.DataFrame(
    scaler.inverse_transform(X_train_knn_imp), columns=X_train.columns
)
X_test_knn_final = pd.DataFrame(
    scaler.inverse_transform(X_test_knn_imp), columns=X_test.columns
)

print("Missing values after KNN:", X_train_knn_final.isnull().sum().sum())
X_train_knn_final.head()


**Parameters:**
- `n_neighbors`: smaller = higher variance, larger = smoother but possibly
  too broad. Try 3, 5, 7 and compare downstream AUC.
- `weights="distance"`: closer neighbors count more than far ones.

**Common mistake:** fitting `StandardScaler` on `X_test` or the full
dataset instead of `X_train` only — same leakage rule as imputers.

**Reflection:**
- Why does KNN imputation get slower as the dataset grows?
- What happens if two rows are similar in age/BMI but very different in
  an unrelated feature?


## Section 8 — Iterative Imputation (MICE)

**Concept (MICE = Multiple Imputation by Chained Equations):**
1. Fill all missing values with a simple starting guess (e.g. mean).
2. For each column with missing values, temporarily mark its imputed
   values as missing again.
3. Regress that column on all other columns (including previously imputed
   ones) and predict the missing values.
4. Cycle through every column with missing values — that's one iteration.
5. Repeat for `max_iter` iterations until estimates stabilize.

**Why it's better for MAR:** it explicitly models relationships between
columns (e.g. glucose ~ income_bracket + other features), unlike a single
global mean.

**Why it's expensive:** each iteration refits a regression model per
column with missing values.


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # required import
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder

enc2 = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
enc2.fit(X_train[cat_cols])

X_train_mice = X_train.copy()
X_test_mice  = X_test.copy()
X_train_mice[cat_cols] = enc2.transform(X_train[cat_cols])
X_test_mice[cat_cols]  = enc2.transform(X_test[cat_cols])

iter_imp = IterativeImputer(max_iter=10, random_state=42, verbose=0)
iter_imp.fit(X_train_mice)

X_train_mice_imp = iter_imp.transform(X_train_mice)
X_test_mice_imp  = iter_imp.transform(X_test_mice)

X_train_mice_df = pd.DataFrame(X_train_mice_imp, columns=X_train.columns)
X_test_mice_df  = pd.DataFrame(X_test_mice_imp,  columns=X_test.columns)

print("Missing after MICE:", X_train_mice_df.isnull().sum().sum())
X_train_mice_df.head()


**Common mistake:** forgetting
`from sklearn.experimental import enable_iterative_imputer` before
importing `IterativeImputer` — without it you'll get an `ImportError`.

**Reflection:**
- If `max_iter=1`, is MICE meaningfully different from simple imputation?
- For a dataset with 50 columns and 40% missingness, would you still
  reach for MICE? What's the tradeoff?


## Section 9 — Missingness Indicator Features

**Concept:** Sometimes *whether* a value is missing is more predictive
than any value you could impute. For MNAR `blood_pressure`, the flag
`bp_missing=1` can encode "this patient avoided a high-risk reading" —
information no imputation method alone can recover.

**Important:** add indicator columns *before* imputing — once you impute,
the information about which values were originally missing is gone unless
you saved it.


In [ ]:
X_train_ind = X_train.copy()
X_test_ind  = X_test.copy()

for col in ["bmi", "blood_pressure", "glucose"]:
    X_train_ind[f"{col}_missing"] = X_train_ind[col].isna().astype(int)
    X_test_ind[f"{col}_missing"]  = X_test_ind[col].isna().astype(int)

# Now impute (mean, for simplicity in this comparison)
mean_imp2 = SimpleImputer(strategy="mean")
mean_imp2.fit(X_train_ind[num_cols])

X_train_ind[num_cols] = mean_imp2.transform(X_train_ind[num_cols])
X_test_ind[num_cols]  = mean_imp2.transform(X_test_ind[num_cols])

X_train_ind.columns.tolist()


**When indicators help:** when missingness is MAR/MNAR and correlates
with the target — likely true for `blood_pressure` here.

**When indicators add noise:** when missingness is MCAR and unrelated to
the target — likely true for `bmi`. With small datasets, extra indicator
columns can add dimensionality without adding signal.

**Tip:** `sklearn.impute.MissingIndicator(features="missing-only")`
automates this — see Section 11's pipeline.


## Section 10 — Model Comparison

**Concept:** Use the *same* split and the *same* model across workflows so
the only thing that changes is the imputation strategy.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import LabelEncoder

def run_workflow(X_tr, X_te, y_tr, y_te, label):
    X_tr_enc = X_tr.copy()
    X_te_enc = X_te.copy()
    for col in X_tr.select_dtypes(include=["object", "category", "string"]).columns:
        le = LabelEncoder()
        X_tr_enc[col] = le.fit_transform(X_tr[col].astype(str))
        X_te_enc[col] = le.transform(X_te[col].astype(str))

    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_tr_enc, y_tr)
    proba = clf.predict_proba(X_te_enc)[:, 1]
    pred  = clf.predict(X_te_enc)

    return {
        "Workflow": label,
        "AUC": round(roc_auc_score(y_te, proba), 3),
        "F1": round(f1_score(y_te, pred), 3),
        "Indicators": "Yes" if any("_missing" in c for c in X_tr.columns) else "No",
    }

results = [
    run_workflow(X_train_mean, X_test_mean, y_train, y_test, "SimpleImputer (mean)"),
    run_workflow(X_train_knn_final, X_test_knn_final, y_train, y_test, "KNNImputer (k=5)"),
    run_workflow(X_train_mice_df, X_test_mice_df, y_train, y_test, "IterativeImputer"),
    run_workflow(X_train_ind, X_test_ind, y_train, y_test, "Mean + Indicators"),
]

results_df = pd.DataFrame(results)
results_df


**Interpretation:**
- With only 3 columns missing at 3–12% rates, expect small differences
  between imputers. At 30–50% missingness, gaps would widen.
- AUC is generally preferable to F1 here since it doesn't depend on a
  classification threshold and handles imbalance better.
- Indicators tend to help most for MNAR/MAR columns whose missingness
  correlates with the target.
- Because `has_condition` was generated independently of every feature,
  expect AUC values to hover near 0.5 regardless of imputer — that's
  correct behavior here, not a bug. The point of this comparison is to
  practice the *workflow*, not to chase a high score.

**Reflection:**
- Why doesn't `IterativeImputer` dramatically beat mean imputation here?
- If you used accuracy instead of AUC, what misleading conclusion might
  you draw given the 75/25 class split?


## Section 11 — Production Pipelines

**Concept:** A `Pipeline` + `ColumnTransformer` bundles every preprocessing
step with the model into one object. Calling `.fit(X_train, y_train)`
fits every internal transformer on training data only — leakage is
structurally prevented.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

num_cols = ["age", "bmi", "blood_pressure", "glucose", "hospital_visits"]
cat_cols = ["smoking_status", "income_bracket"]

# Numeric branch: impute (MICE) -> scale
num_pipeline = Pipeline([
    ("imputer", IterativeImputer(max_iter=10, random_state=42)),
    ("scaler",  StandardScaler()),
])

# Categorical branch: impute most-frequent -> one-hot encode
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Indicator branch: missingness flags, only for columns that actually have gaps
indicator_pipeline = MissingIndicator(features="missing-only")

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
    ("ind", indicator_pipeline, num_cols),
])

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",  LogisticRegression(max_iter=500, random_state=42)),
])

full_pipeline.fit(X_train, y_train)
auc = roc_auc_score(y_test, full_pipeline.predict_proba(X_test)[:, 1])
print(f"Pipeline AUC: {auc:.3f}")


**Why pipelines prevent leakage:** every transformer fits only on
whatever data is passed to `.fit()`. In cross-validation
(`cross_val_score(pipeline, X, y)`), the pipeline refits from scratch on
each fold's training data — no leakage across folds either.


## Section 12 — Saving Artifacts

**Concept:** A fitted imputer is a trained artifact, just like a model.
If you refit it on new data at inference time instead of loading the
saved one, your production feature distribution silently drifts away
from what the model was trained on (training-serving skew).


In [ ]:
import joblib

# Save the complete pipeline (imputers, scaler, encoder, model — all in one)
joblib.dump(full_pipeline, "pipeline_v1.joblib")

# Save individual imputers separately if needed
joblib.dump(mean_imp, "mean_imputer.joblib")
joblib.dump(knn_imp,  "knn_imputer.joblib")
joblib.dump(iter_imp, "mice_imputer.joblib")

# Loading for inference — never refit on new data
loaded_pipeline = joblib.load("pipeline_v1.joblib")
new_predictions = loaded_pipeline.predict_proba(X_test)[:, 1]
print(new_predictions[:5].round(3))


**Common mistake:** saving only the model weights and discarding the
preprocessing pipeline — at serving time you'd have to manually
reimplement imputation and scaling, risking subtle mismatches.


## Section 13 — Final Deliverables

### Missingness summary table

| Feature | Missing % | Suspected Type | Evidence |
|---|---|---|---|
| bmi | ~12% | MCAR | Equal missingness rate across groups; near-zero correlation with other features |
| glucose | ~8% | MAR | ~35% missing for low income vs ~7% for high income |
| blood_pressure | ~3% | MNAR | Missingness concentrated where proxy indicators suggest high-BP risk |

### Imputation decision table

| Feature | Strategy | Reason |
|---|---|---|
| bmi | Mean | MCAR + roughly normal distribution |
| blood_pressure | IterativeImputer + indicator | MNAR; preserves relationships; flag captures missingness signal |
| glucose | IterativeImputer | MAR; income_bracket as a predictor captures the pattern |
| smoking_status | Most frequent | Categorical, no ordinal relationship |
| income_bracket | Most frequent | Categorical, no ordinal relationship |
| hospital_visits | Median | Right-skewed count data |

### Model comparison table

Fill this in with your actual run's numbers from Section 10's `results_df`.

| Workflow | AUC | F1 | Indicators Used |
|---|---|---|---|
| SimpleImputer (mean) | — | — | No |
| KNNImputer (k=5) | — | — | No |
| IterativeImputer | — | — | No |
| Mean + Indicators | — | — | Yes |

### Production recommendation

Use the full sklearn `Pipeline` with `IterativeImputer` for numeric
columns, `most_frequent` for categoricals, and `MissingIndicator` flags
for MNAR/MAR columns. Serialize the entire pipeline with `joblib`. Never
refit imputers at serving time. Monitor feature drift in production,
especially for `blood_pressure`.


## Section 14 — Reflection Questions

**1. Why does mean imputation potentially make a model overconfident?**
It replaces every missing value with the same number, artificially
shrinking variance. The model sees a false concentration of "average"
rows and underestimates how uncertain predictions should be for true
outliers.

**2. What information leaks when an imputer is fitted before splitting?**
The test set's statistics (mean, nearest neighbors, etc.) influence the
values imputed into the training set, giving training data a preview of
the test distribution. Evaluation metrics become optimistic versus
real-world performance.

**3. Why is MNAR difficult to handle?**
Because the missingness depends on the very value that's missing — you
can't observe that relationship directly. Any standard imputation method
introduces some bias; domain knowledge and sensitivity analysis are the
best available tools.

**4. How does KNNImputer handle categorical variables?**
It doesn't, natively — it relies on Euclidean distance, which needs
numeric input. Categoricals must be ordinally encoded first, and imputed
values come back as numeric codes that need mapping back to categories.

**5. Why should indicator columns be added before imputation?**
Once you impute, the information that a value was ever missing disappears
unless you've captured it separately. Adding the flag first preserves
that signal as its own feature.

**6. When would you choose not to impute at all?**
When missingness is very high (60–70%+) for a column, when the column is
strongly MNAR with no usable proxy, when the column isn't important to
the task, or when using models (like gradient-boosted trees) that handle
NaN natively.


## Section 15 — Stretch Challenges

Try these on your own. Compare results rather than looking for one
"correct" answer.

1. **Increase missingness to 50%** — change the bmi mask threshold from
   `0.12` to `0.5`. Does the gap between mean imputation and
   `IterativeImputer` widen?
2. **Add a missing categorical** — introduce `NaN` into `smoking_status`
   for 15% of rows and compare imputation approaches.
3. **Mean vs median for `hospital_visits`** — plot the distribution before
   and after each.
4. **Vary `k` in `KNNImputer`** — try 1, 3, 5, 10, 20 and plot AUC vs k.
5. **Swap in `RandomForestClassifier`** — does imputation strategy matter
   more or less for tree-based models?
6. **Indicators vs no indicators ablation** — build a full grid of results
   across imputers × indicator on/off.
7. **Time-based missingness** — if your data had timestamps, simulate
   missingness that increases over time (sensor drift) and think through
   how you'd detect and handle it differently from static MCAR.
